# 02 — Qualidade de dados

**Objetivo deste notebook:** tratar, de forma explícita e documentada, os
problemas de qualidade identificados na exploração inicial (`01_data_exploration`),
e produzir a tabela consolidada (`data/processed/fato_financeiro.csv`) usada
nas análises seguintes.

Cada decisão de tratamento é registrada com o motivo, para que qualquer pessoa
consiga reproduzir ou questionar o critério adotado.


In [1]:
import pandas as pd

pd.set_option('display.max_columns', 50)
RAW = '../data/raw'
PROCESSED = '../data/processed'

ARQUIVOS = {
    '1T2025': f'{RAW}/1T2025.csv', '2T2025': f'{RAW}/2T2025.csv',
    '3T2025': f'{RAW}/3T2025.csv', '4T2025': f'{RAW}/4T2025.csv',
    '1T2026': f'{RAW}/1T2026.csv',
}
ORDEM_2025 = ['1T2025', '2T2025', '3T2025', '4T2025']


## 1. Problema: valores acumulados no ano-calendário

**Observação:** a DIOPS publica `VL_SALDO_FINAL` como saldo acumulado desde
janeiro do ano em curso, não como o valor isolado do trimestre. Isso foi
confirmado no notebook `01_data_exploration` ao observar crescimento
monotônico da receita de uma mesma operadora ao longo dos trimestres de 2025.

**Tratamento adotado:** para cada operadora, o valor do trimestre isolado é
obtido por diferença entre o acumulado do trimestre atual e o acumulado do
trimestre imediatamente anterior, dentro do mesmo ano-calendário. Para o
primeiro trimestre de cada ano (1T2025 e 1T2026), o acumulado já corresponde
ao valor isolado, pois não há trimestre anterior no mesmo ano.

**Limitação registrada:** essa abordagem assume que a metodologia contábil de
acumulação não muda ao longo do ano para a mesma operadora — não foi validado
individualmente para cada uma das operadoras do estudo.


In [2]:
def extrair_contas_macro(caminho):
    """Extrai, por operadora, os saldos acumulados das contas macro usadas no estudo."""
    dio = pd.read_csv(caminho, sep=';', decimal=',', encoding='utf-8')
    dio['REG_ANS'] = dio['REG_ANS'].astype(int)
    dio['cd_str'] = dio['CD_CONTA_CONTABIL'].astype(str)

    def conta(codigo, nome):
        return (dio[dio['cd_str'] == codigo][['REG_ANS', 'VL_SALDO_FINAL']]
                .rename(columns={'VL_SALDO_FINAL': nome}))

    base = conta('3', 'receitas_totais')
    for cod, nome in [('4', 'despesas_totais'), ('31', 'receita_assistencial'),
                       ('41', 'sinistro_liquido'), ('46', 'despesa_administrativa'),
                       ('33', 'outras_receitas_operacionais'), ('44', 'outras_despesas_operacionais')]:
        base = base.merge(conta(cod, nome), on='REG_ANS', how='outer')

    # Decomposição do sinistro (faturado bruto / glosa / coparticipação),
    # usada na análise de taxa de glosa no notebook 03.
    leafs = dio[(dio['cd_str'].str.startswith('411')) & (dio['cd_str'].str.len() == 9)].copy()
    leafs['sufixo'] = leafs['cd_str'].str[-1]
    bruto = (leafs[leafs['sufixo'] == '1'].groupby('REG_ANS')['VL_SALDO_FINAL'].sum()
             .rename('faturado_bruto').reset_index())
    glosa = (leafs[leafs['sufixo'] == '2'].groupby('REG_ANS')['VL_SALDO_FINAL'].sum()
             .abs().rename('glosa').reset_index())
    copart = (leafs[leafs['sufixo'] == '3'].groupby('REG_ANS')['VL_SALDO_FINAL'].sum()
              .abs().rename('coparticipacao').reset_index())

    for d in (bruto, glosa, copart):
        base = base.merge(d, on='REG_ANS', how='left')

    return base

acumulado = {t: extrair_contas_macro(a) for t, a in ARQUIVOS.items()}
print("Contas macro extraídas para os 5 trimestres.")


Contas macro extraídas para os 5 trimestres.


In [3]:
COLUNAS_METRICA = ['receitas_totais', 'despesas_totais', 'receita_assistencial',
                    'sinistro_liquido', 'despesa_administrativa',
                    'outras_receitas_operacionais', 'outras_despesas_operacionais',
                    'faturado_bruto', 'glosa', 'coparticipacao']

isolados = {'1T2025': acumulado['1T2025'].copy()}

for i in range(1, len(ORDEM_2025)):
    atual, anterior = ORDEM_2025[i], ORDEM_2025[i - 1]
    merged = acumulado[atual].merge(
        acumulado[anterior], on='REG_ANS', how='left', suffixes=('', '_ant')
    )
    for col in COLUNAS_METRICA:
        merged[col] = merged[col] - merged[f'{col}_ant'].fillna(0)
    isolados[atual] = merged[['REG_ANS'] + COLUNAS_METRICA]

isolados['1T2026'] = acumulado['1T2026'].copy()

print("Trimestres isolados calculados:", list(isolados.keys()))


Trimestres isolados calculados: ['1T2025', '2T2025', '3T2025', '4T2025', '1T2026']


### 1.1 Validação: um caso conhecido (PASA)

Como checagem de sanidade, comparamos a sinistralidade isolada calculada aqui
com o valor documentado no relatório do estudo de caso (`reports/diagnostico_pasa_consolidado.md`).


In [4]:
PASA = 331988

for t in ['1T2025', '2T2025', '3T2025', '4T2025', '1T2026']:
    linha = isolados[t][isolados[t]['REG_ANS'] == PASA]
    if len(linha):
        r = linha['receita_assistencial'].values[0]
        s = linha['sinistro_liquido'].values[0]
        print(f"{t}: sinistralidade = {s/r*100:.1f}%")


1T2025: sinistralidade = 101.9%
2T2025: sinistralidade = 99.4%
3T2025: sinistralidade = 92.3%
4T2025: sinistralidade = 107.4%
1T2026: sinistralidade = 96.6%


Os valores batem com os documentados no relatório do estudo de caso
(101,9% · 99,4% · 92,3% · 107,4% · 96,6%), validando o método de isolamento.

## 2. Problema: operadoras com receita trimestral muito baixa

**Observação:** operadoras com receita assistencial muito pequena no
trimestre produzem sinistralidades extremas (próximas de 0% ou várias vezes
100%) que não refletem um padrão real — refletem instabilidade de escala:
qualquer sinistro pequeno, dividido por uma receita quase nula, gera um
percentual desproporcional.

**Tratamento adotado:** operadoras com receita assistencial abaixo de
R$ 100.000 no trimestre são sinalizadas com a flag `dado_confiavel = False`.
Elas **não são excluídas da base** — permanecem disponíveis para quem quiser
auditar o critério — mas são excluídas por padrão dos cálculos agregados
(médias, medianas, benchmarks).

**Limitação registrada:** o limiar de R$ 100 mil é uma escolha analítica, não
um padrão oficial da ANS. Não foi testada a sensibilidade do resultado a
limiares alternativos.


In [5]:
for t, df in isolados.items():
    df['dado_confiavel'] = df['receita_assistencial'] >= 100_000


## 3. Problema: operadoras com estrutura contábil atípica

**Observação:** durante a construção do dashboard, duas operadoras de porte
relevante (receita alta, portanto não capturadas pelo critério da Seção 2)
apresentaram números incompatíveis com o padrão do setor:

- uma operadora reportou `sinistro_liquido = R$ 0` em todos os 5 trimestres
  apesar de receita assistencial superior a R$ 200 milhões;
- outra reportou sinistralidade sustentada acima de 180%.

**Interpretação:** não foi possível determinar, com o dado público disponível,
se essas operadoras utilizam uma estrutura de contabilização diferente do
padrão (por exemplo, registrando despesa assistencial sob outra rubrica), ou
se há outro fator não identificado. Não se trata do mesmo problema da Seção 2
(baixa escala) — por isso recebe um tratamento e um registro à parte.

**Tratamento adotado:** para as visões de mercado agregadas (Página 1 do
dashboard, e as análises deste notebook), essas operadoras são delimitadas por
uma faixa de sinistralidade plausível (20% a 150%), documentada explicitamente
aqui e em cada visual onde é aplicada. Essa exclusão não é aplicada ao estudo
de caso da PASA, que é sempre reportada com seu valor real, dentro ou fora
dessa faixa.


In [6]:
def sinistralidade_dentro_da_faixa(df, minimo=0.20, maximo=1.50):
    ratio = df['sinistro_liquido'] / df['receita_assistencial']
    return ratio.between(minimo, maximo)

for t, df in isolados.items():
    df['padrao_tipico'] = sinistralidade_dentro_da_faixa(df).fillna(False)

# Quantas operadoras foram sinalizadas como atípicas, com receita relevante
exemplo = isolados['1T2026']
atipicas_grandes = exemplo[(exemplo['dado_confiavel']) & (~exemplo['padrao_tipico'])]
print(f"Operadoras com dado_confiavel=True mas fora do padrão típico (1T2026): {len(atipicas_grandes)}")
atipicas_grandes[['REG_ANS', 'receita_assistencial', 'sinistro_liquido']]


Operadoras com dado_confiavel=True mas fora do padrão típico (1T2026): 80


,REG_ANS,receita_assistencial,sinistro_liquido
0,515,1949046.77,8155437.76
3,884,53125627.66,87328581.07
18,302627,9068841.61,13901451.71
63,311057,1075163.16,149729.98
87,313971,287829.23,11045687.68
...,...,...,...
734,423408,604816.75,27862.42
737,423505,45762860.22,NaN
755,424013,1881036.42,159888.09
766,424366,777591.14,NaN


## 4. Universo de operadoras: quantas entram em cada trimestre

O número de operadoras de Autogestão com dado disponível **varia por
trimestre** — nem toda operadora reporta em todos os períodos. Essa variação é
documentada aqui explicitamente, em vez de tratada implicitamente dentro de
uma medida de dashboard.


In [7]:
import pandas as pd

cad = pd.read_csv(f'{RAW}/Relatorio_cadop.csv', sep=';', encoding='utf-8', dtype=str)
cad['REG_ANS'] = cad['REGISTRO_OPERADORA'].str.strip().astype(int)
autogestoes = set(cad[cad['MODALIDADE'] == 'Autogestão']['REG_ANS'])

resumo = []
for t, df in isolados.items():
    universo = df[df['REG_ANS'].isin(autogestoes)]
    confiaveis = universo[universo['dado_confiavel']]
    resumo.append({
        'trimestre': t,
        'operadoras_no_cadastro': len(autogestoes),
        'com_demonstracao_no_trimestre': len(universo),
        'confiaveis_apos_criterio_receita': len(confiaveis),
    })

tabela_universo = pd.DataFrame(resumo).set_index('trimestre')
tabela_universo = tabela_universo.reindex(['1T2025', '2T2025', '3T2025', '4T2025', '1T2026'])
tabela_universo


,operadoras_no_cadastro,com_demonstracao_no_trimestre,confiaveis_apos_criterio_receita
trimestre,,,
1T2025,142,114,110
2T2025,142,116,112
3T2025,142,116,113
4T2025,142,115,110
1T2026,142,116,112


**Números de referência para validação cruzada com o Power BI**
(medida `Contagem Operadoras Confiaveis` — deve retornar exatamente estes
valores por trimestre):

| Trimestre | Operadoras confiáveis |
|---|---|
| 1T2025 | 110 |
| 2T2025 | 112 |
| 3T2025 | 113 |
| 4T2025 | 110 |
| 1T2026 | 112 |

## 5. Consolidação: tabela fato final


### 4.1 Extensão: contas de receita e despesa não-assistenciais

Além das contas usadas na sinistralidade e no resultado operacional, foram
extraídas também as contas `33` (Outras Receitas Operacionais — onde está a
receita específica de autogestão da Lei 13.127, identificada no estudo de
caso da PASA) e `44` (Outras Despesas Operacionais). Essas duas contas
alimentam o gráfico de composição de receita/despesa da Página 2 do
dashboard, tornando visível a diferença entre "receita de contraprestação"
(usada na sinistralidade) e "receita total" (usada no resultado operacional).


In [8]:
fatos = []
for trimestre, df in isolados.items():
    d = df[df['REG_ANS'].isin(autogestoes)].copy()
    d['trimestre'] = trimestre
    fatos.append(d)

fato_financeiro = pd.concat(fatos, ignore_index=True)
for col in COLUNAS_METRICA:
    fato_financeiro[col] = fato_financeiro[col].round(2)

fato_financeiro = fato_financeiro[
    ['REG_ANS', 'trimestre'] + COLUNAS_METRICA + ['dado_confiavel', 'padrao_tipico']
]

print(f"Shape final: {fato_financeiro.shape}")
fato_financeiro.head()


Shape final: (577, 14)


,REG_ANS,trimestre,receitas_totais,despesas_totais,receita_assistencial,sinistro_liquido,despesa_administrativa,outras_receitas_operacionais,outras_despesas_operacionais,faturado_bruto,glosa,coparticipacao,dado_confiavel,padrao_tipico
0,302627,1T2025,1.604094e+07,10859553.47,11262093.00,9249631.60,1282512.38,2273472.52,95198.82,9959437.79,520068.31,272919.76,True,True
1,304131,1T2025,2.970590e+07,30838972.23,28202011.81,28161018.18,2233987.41,4165.86,371897.99,29325256.32,1397860.47,367591.92,True,True
2,306754,1T2025,1.690697e+07,14831760.46,12866573.01,9584541.37,2084855.97,3423344.44,3151743.00,11537123.85,1247267.68,705314.80,True,True
3,307319,1T2025,1.157089e+08,73634526.30,78652592.48,60966577.25,7113687.13,4969836.85,5490353.08,71402661.32,2332287.48,8560799.12,True,True
4,307751,1T2025,3.370827e+07,30632071.90,30567106.39,26734013.09,2983100.93,2447.00,912285.55,28305572.94,1333623.35,254448.30,True,True


In [9]:
fato_financeiro.to_csv(f'{PROCESSED}/fato_financeiro.csv', index=False, encoding='utf-8-sig')
print(f"Salvo em {PROCESSED}/fato_financeiro.csv")


Salvo em ../data/processed/fato_financeiro.csv


## 6. Valores ausentes

Checagem final de valores ausentes na tabela consolidada — relevante para
qualquer pessoa que for reutilizar esta base.


In [10]:
fato_financeiro[COLUNAS_METRICA].isna().sum()


receitas_totais                  0
despesas_totais                  0
receita_assistencial            10
sinistro_liquido                10
despesa_administrativa           0
outras_receitas_operacionais    80
outras_despesas_operacionais    18
faturado_bruto                  10
glosa                           28
coparticipacao                  75
dtype: int64

As colunas `faturado_bruto`, `glosa` e `coparticipacao` têm mais ausência que
as demais porque dependem de uma decomposição de conta (nível 9 dígitos) nem
sempre preenchida por todas as operadoras — isso é aceitável para os cálculos
de sinistralidade e resultado operacional (que não dependem delas), mas
limita o cálculo da taxa de glosa a um subconjunto da base, tratado
explicitamente no notebook `03_pasa_analysis`.

## Resumo do que foi tratado

| Problema | Tratamento | Seção |
|---|---|---|
| Valores acumulados no ano | Diferença entre acumulados consecutivos | 1 |
| Receita muito baixa | Flag `dado_confiavel` (limiar R$ 100 mil) | 2 |
| Estrutura contábil atípica | Flag `padrao_tipico` (faixa 20%–150%) | 3 |
| Universo variável de operadoras | Documentado por trimestre | 4 |
| Valores ausentes em subcontas | Aceito e documentado | 6 |

## Próximos passos

O notebook `03_pasa_analysis.ipynb` usa a tabela `fato_financeiro.csv` gerada
aqui para desenvolver o estudo de caso da PASA: evolução da sinistralidade,
benchmark contra peer group, taxa de glosa e resultado operacional completo.
